# Lab 01 — Konsep Dasar Apache Spark
**Arsitektur, partition, lazy evaluation, DAG, job/stage/task, dan Spark UI**

Notebook ini dirancang untuk environment pelatihan: Spark 3.5.7 standalone, Jupyter, 2 worker, masing-masing 2 core / 2 GB.  
**Prinsip lab:** prediksi → jalankan → amati → jelaskan. Jangan hanya mengejar output akhir.

## Capaian praktik
- Mengidentifikasi Driver, Master/Cluster Manager, Worker, Executor, Partition, Task, Stage, dan Job.
- Membuktikan lazy evaluation dan membedakan transformation vs action.
- Menghubungkan `explain()` dengan Jobs/Stages/Tasks di Spark UI.
- Mengamati pengaruh jumlah partition terhadap paralelisme.

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T
import time, json, os, shutil

spark = (
    SparkSession.builder
    .appName("spark-training-lab")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 02:49:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 3.5.7
Master        : spark://spark-master:7077
App name      : spark-training-lab
Default parallelism: 2


## Lab 1 — Kenali cluster yang sedang dipakai (15–20 menit)
**Sebelum menjalankan kode, prediksi:** berapa core total cluster dan di mana Driver berjalan?

Buka juga:
- Master UI: http://localhost:8080
- Worker UI: http://localhost:8081 dan http://localhost:8082
- Application UI: http://localhost:4040 (selama SparkSession aktif)

Catat jumlah worker, core, memory, dan executor yang terlihat.

In [2]:
print("Application ID:", spark.sparkContext.applicationId)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("UI URL dari SparkContext:", spark.sparkContext.uiWebUrl)

Application ID: app-20261110024957-0010
Default parallelism: 4
UI URL dari SparkContext: http://c6489f47d953:4040


**Observasi peserta**
1. Apakah jumlah core cluster sama dengan jumlah core yang digunakan aplikasi?
2. Apakah Driver merupakan Worker?
3. Mengapa aplikasi kedua dapat menunggu resource walaupun cluster masih hidup?

## Lab 2 — Partition → Task (20 menit)
Buat DataFrame dengan 4 partition, lalu ubah menjadi 8. Amati `count()` pada Spark UI.

In [3]:
df4 = spark.range(0, 1_000_000, 1, 4)
df8 = df4.repartition(8)

print("df4 partitions:", df4.rdd.getNumPartitions())
print("df8 partitions:", df8.rdd.getNumPartitions())
print("count:", df4.count())

df4 partitions: 4


[Stage 0:=============================>                             (2 + 2) / 4]

df8 partitions: 8
count: 1000000


In [ ]:
# Lihat jumlah record per partition.
partition_profile = (
    df8.rdd
    .mapPartitionsWithIndex(lambda idx, it: [(idx, sum(1 for _ in it))])
    .collect()
)
partition_profile

**Challenge:** Jika stage memiliki 8 task tetapi hanya 4 slot task tersedia, berapa gelombang minimal yang dibutuhkan bila durasi task seragam? Jelaskan hubungan partition, task, dan core.

## Lab 3 — Lazy evaluation, transformation, action (20 menit)
Jalankan cell pertama dan lihat Spark UI. Apakah job baru muncul? Baru setelah itu jalankan action.

In [ ]:
base = spark.range(0, 2_000_000, 1, 4)
transformed = (
    base
    .filter((F.col("id") % 2) == 0)
    .withColumn("bucket", F.col("id") % 10)
    .select("id", "bucket")
)
print("DataFrame sudah didefinisikan. Belum ada action eksplisit.")

In [ ]:
# ACTION
result_count = transformed.count()
print("Rows:", result_count)

**Pertanyaan:** `filter`, `withColumn`, `select`, `count` termasuk transformation atau action? Apa bukti yang terlihat pada UI?

## Lab 4 — Dari plan ke shuffle (25 menit)
`groupBy` adalah wide transformation. Cari `Exchange` pada physical plan dan cocokkan dengan stage boundary di UI.

In [ ]:
agg = (
    transformed
    .groupBy("bucket")
    .agg(F.count("*").alias("n"), F.sum("id").alias("sum_id"))
    .orderBy("bucket")
)
agg.explain("formatted")

In [ ]:
agg.show(truncate=False)

**Checklist Spark UI:**
- Job mana yang muncul karena `show()`?
- Berapa stage?
- Stage mana yang memiliki shuffle read/write?
- Berapa task pada masing-masing stage?

Jangan menganggap satu pemanggilan API = satu task.

## Lab 5 — Eksperimen jumlah partition (30 menit)
Ini **bukan benchmark ilmiah**. Tujuannya melihat trade-off parallelism vs scheduling overhead.

In [ ]:
rows = []
for n in [1, 2, 4, 8]:
    sample = spark.range(0, 10_000_000, 1, n)
    t0 = time.perf_counter()
    total = sample.agg(F.sum("id")).collect()[0][0]
    elapsed = time.perf_counter() - t0
    rows.append((n, round(elapsed, 3), total))

spark.createDataFrame(rows, ["partitions", "seconds", "sum_id"]).show(truncate=False)

**Expected checksum:** `49,999,995,000,000`.

Diskusikan:
- Mengapa 8 partition tidak selalu 8× lebih cepat daripada 1 partition?
- Apa yang terjadi jika jumlah task jauh lebih banyak daripada core?
- Mengapa pengukuran perlu diulang?

## Exit ticket
Tulis jawaban singkat:
1. Worker, Executor, Task, Partition: apa bedanya?
2. Apa yang memicu Job?
3. Mengapa `Exchange` penting saat membaca plan?
4. Mengapa `repartition()` bukan optimasi gratis?